# 12 · TTF Market Analysis

Classic natural gas price and forward curve analysis: volatility surface, curve shape, seasonal carry, and risk metrics.

**Run order:** Notebook 01 (storage data) then 07 or this standalone if `ttf_curve.csv` already exists.

In [ ]:
import sys, os, warnings
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from scipy import stats
from pathlib import Path
from datetime import date
import calendar
warnings.filterwarnings('ignore')

for _c in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    if (_c / 'src' / 'agsi_client').exists():
        sys.path.insert(0, str(_c)); os.chdir(_c)
        print(f"\u2705 Root: {_c}"); break

# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550
# CONFIGURATION
# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550
ANALYSIS_DATE = None          # None = last data point | or date(2024,9,1)
START_DATE    = "2019-01-01"
# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550

ttf_path     = Path("data/raw/ttf_curve.csv")
storage_path = Path("data/processed/eu_aggregate_full.parquet")
lng_path     = Path("data/processed/eu_lng_full.parquet")

ttf_df     = pd.read_csv(ttf_path, index_col=0, parse_dates=True) if ttf_path.exists() else pd.DataFrame()
storage_df = pd.read_parquet(storage_path) if storage_path.exists() else pd.DataFrame()
lng_df     = pd.read_parquet(lng_path) if lng_path.exists() else pd.DataFrame()

# Strip timezone info \u2014 TTF CSV is tz-aware (UTC), parquet files are tz-naive
for _df in [ttf_df, storage_df, lng_df]:
    try:
        _df.index = pd.to_datetime(_df.index).tz_localize(None)
    except Exception:
        pass

if not ttf_df.empty:
    ttf_df.columns = [c.strip() for c in ttf_df.columns]
    for col in ttf_df.columns:
        ttf_df[col] = pd.to_numeric(ttf_df[col], errors='coerce')
    ttf_df = ttf_df[ttf_df.index >= START_DATE].sort_index()
if not storage_df.empty:
    storage_df = storage_df[storage_df.index >= START_DATE].sort_index()
if not lng_df.empty:
    lng_df = lng_df[lng_df.index >= START_DATE].sort_index()

analysis_date = (
    ANALYSIS_DATE if ANALYSIS_DATE
    else (ttf_df.index[-1].date() if not ttf_df.empty else date.today())
)
print(f"Analysis date : {analysis_date}")
print(f"TTF rows      : {len(ttf_df)}")
print(f"Storage rows  : {len(storage_df)}")
print(f"LNG rows      : {len(lng_df)}" + (" (loaded)" if not lng_df.empty else " (not found \u2014 LNG section will be skipped)"))


## 1 · Price History & Regimes

TTF M1 price history annotated with key market events. YoY % change highlights the 2022 energy crisis amplitude and the post-crisis normalisation.

In [ ]:
if ttf_df.empty:
    print("\u26a0\ufe0f  No TTF data. Run notebook 01 / 07 first.")
else:
    s = ttf_df["M1"].dropna()
    EVENTS = {
        "COVID low\n(Apr 2020)": ("2020-04-01", "#16a34a"),
        "Crisis peak\n(Aug 2022)": ("2022-08-26", "#dc2626"),
        "Post-crisis norm\n(Jul 2023)": ("2023-07-01", "#2563eb"),
    }

    fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                        subplot_titles=["TTF M1 Price (\u20ac/MWh)", "YoY Price Change (%)"],
                        vertical_spacing=0.08, row_heights=[0.65, 0.35])

    fig.add_trace(go.Scatter(x=s.index, y=s.values, name="M1 Price",
                             line=dict(color="#6366f1", width=1.5)), row=1, col=1)

    for label, (dt_str, color) in EVENTS.items():
        try:
            dt = pd.Timestamp(dt_str)
            if s.index.min() <= dt <= s.index.max():
                pv = float(s.asof(dt))
                fig.add_vline(x=dt.timestamp() * 1000, line_color=color, line_dash="dash",
                              line_width=1, row=1, col=1)
                fig.add_annotation(x=dt, y=pv, text=label, showarrow=True,
                                   arrowhead=2, arrowcolor=color,
                                   font=dict(size=8, color=color), row=1, col=1)
        except Exception:
            pass

    yoy = s.pct_change(252) * 100
    fig.add_trace(go.Scatter(x=yoy.index, y=yoy.clip(lower=0), fill="tozeroy", name="YoY +",
                             line=dict(color="#22c55e", width=0.5),
                             fillcolor="rgba(34,197,94,0.3)"), row=2, col=1)
    fig.add_trace(go.Scatter(x=yoy.index, y=yoy.clip(upper=0), fill="tozeroy", name="YoY \u2212",
                             line=dict(color="#ef4444", width=0.5),
                             fillcolor="rgba(239,68,68,0.3)"), row=2, col=1)
    fig.add_hline(y=0, line_color="black", line_width=0.8, row=2, col=1)

    fig.update_yaxes(title_text="\u20ac/MWh", row=1, col=1)
    fig.update_yaxes(title_text="YoY %", row=2, col=1)
    fig.update_layout(height=580, template="plotly_white",
                      title="TTF M1 Price History with Key Events",
                      legend=dict(orientation="h", y=-0.12))
    fig.show()

    latest = float(s.iloc[-1])
    print(f"  Current M1 price: \u20ac{latest:.2f}/MWh")
    for lag_label, days in [("1yr ago", 252), ("2yr ago", 504)]:
        try:
            ref = float(s.iloc[-days])
            print(f"  {lag_label}       : \u20ac{ref:.2f}/MWh  |  \u0394 = {(latest/ref - 1)*100:+.1f}%")
        except IndexError:
            pass


## 2 · Volatility Surface

Rolling realised volatility at 5d, 21d, 63d and 252d windows. Vol-of-vol captures the uncertainty around uncertainty. Regime chart colour-codes each day by vol regime.

In [ ]:
if ttf_df.empty:
    print("\u26a0\ufe0f  No TTF data.")
else:
    s = ttf_df["M1"].dropna()
    log_ret = np.log(s / s.shift(1)).dropna()

    windows = [5, 21, 63, 252]
    labels  = {5: "5d", 21: "21d", 63: "63d", 252: "252d"}
    vols = pd.DataFrame(index=s.index)
    for w in windows:
        vols[f"vol_{w}d"] = log_ret.rolling(w).std() * np.sqrt(252) * 100
    vols["vol_of_vol"] = vols["vol_21d"].rolling(21).std()

    cur21 = float(vols["vol_21d"].dropna().iloc[-1])
    pct21 = float(stats.percentileofscore(vols["vol_21d"].dropna(), cur21))

    palette = {5: "#ef4444", 21: "#f97316", 63: "#3b82f6", 252: "#8b5cf6"}
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                        subplot_titles=["Rolling Realized Volatility (annualised %)",
                                        "Vol of Vol \u2014 21d std of 21d vol (%)"],
                        vertical_spacing=0.08, row_heights=[0.65, 0.35])

    for w in windows:
        fig.add_trace(go.Scatter(x=vols.index, y=vols[f"vol_{w}d"],
                                 name=labels[w], line=dict(color=palette[w], width=1.2)), row=1, col=1)
    fig.add_trace(go.Scatter(x=vols.index, y=vols["vol_of_vol"],
                             fill="tozeroy", name="Vol of Vol",
                             line=dict(color="#64748b", width=1),
                             fillcolor="rgba(100,116,139,0.15)"), row=2, col=1)

    fig.update_yaxes(title_text="Vol %", row=1, col=1)
    fig.update_yaxes(title_text="%", row=2, col=1)
    fig.update_layout(height=540, template="plotly_white",
                      title=f"TTF M1 Volatility \u2014 21d vol: {cur21:.1f}% (p{pct21:.0f} vs history)",
                      legend=dict(orientation="h", y=-0.12))
    fig.show()

    # Vol regime chart
    thresholds   = [0, 40, 80, 150, 9999]
    regime_names = ["Low (<40%)", "Medium (40\u201380%)", "High (80\u2013150%)", "Extreme (>150%)"]
    regime_colors = ["#22c55e", "#f59e0b", "#ef4444", "#7c3aed"]
    vol21 = vols["vol_21d"].dropna()

    fig2 = go.Figure()
    fig2.add_trace(go.Scatter(x=vol21.index, y=vol21, mode="lines",
                              line=dict(color="rgba(0,0,0,0.15)", width=0.8), showlegend=False))
    for i, (name, color) in enumerate(zip(regime_names, regime_colors)):
        mask = (vol21 >= thresholds[i]) & (vol21 < thresholds[i + 1])
        if mask.any():
            fig2.add_trace(go.Scatter(x=vol21[mask].index, y=vol21[mask],
                                      mode="markers",
                                      marker=dict(color=color, size=3, opacity=0.8),
                                      name=name))
    fig2.update_layout(template="plotly_white", height=380,
                       title="Vol Regime \u2014 21d Rolling Volatility Color-Coded",
                       yaxis_title="Vol %", legend=dict(orientation="h", y=-0.15))
    fig2.show()

    print("\nCurrent vol percentiles:")
    for w in windows:
        cur = float(vols[f"vol_{w}d"].dropna().iloc[-1])
        pct = float(stats.percentileofscore(vols[f"vol_{w}d"].dropna(), cur))
        print(f"  {labels[w]:>4}: {cur:5.1f}%  (p{pct:.0f})")


## 3 · Forward Curve Shape Analysis

Curve slope (M1–M12) measures overall backwardation/contango. Curvature (M6 − (M1+M12)/2) captures the belly. Multi-date overlay shows how the curve has shifted over time.

In [ ]:
if ttf_df.empty:
    print("\u26a0\ufe0f  No TTF data.")
else:
    m_cols = sorted([c for c in ttf_df.columns if c.startswith("M") and c[1:].isdigit()],
                    key=lambda c: int(c[1:]))
    m12 = m_cols[:12]

    # Curve slope and curvature history
    has_shape = all(c in ttf_df.columns for c in ["M1", "M6", "M12"])
    if has_shape:
        curve_df = pd.DataFrame({
            "slope":     ttf_df["M1"] - ttf_df["M12"],
            "curvature": ttf_df["M6"] - (ttf_df["M1"] + ttf_df["M12"]) / 2,
        }).dropna()

        fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                            subplot_titles=["Curve Slope: M1 \u2013 M12 (\u20ac/MWh)  \u2192  + = backwardation",
                                            "Curve Curvature: M6 \u2013 (M1+M12)/2 (\u20ac/MWh)  \u2192  + = hump"],
                            vertical_spacing=0.08)
        fig.add_trace(go.Scatter(x=curve_df.index, y=curve_df["slope"],
                                 fill="tozeroy", name="Slope",
                                 line=dict(color="#2E75B6", width=1.5),
                                 fillcolor="rgba(46,117,182,0.12)"), row=1, col=1)
        fig.add_hline(y=0, line_color="black", line_width=0.8, row=1, col=1)
        fig.add_trace(go.Scatter(x=curve_df.index, y=curve_df["curvature"],
                                 fill="tozeroy", name="Curvature",
                                 line=dict(color="#f97316", width=1.5),
                                 fillcolor="rgba(249,115,22,0.12)"), row=2, col=1)
        fig.add_hline(y=0, line_color="black", line_width=0.8, row=2, col=1)
        fig.update_yaxes(title_text="\u20ac/MWh", row=1, col=1)
        fig.update_yaxes(title_text="\u20ac/MWh", row=2, col=1)
        fig.update_layout(height=520, template="plotly_white",
                          title="TTF Forward Curve Shape Metrics",
                          legend=dict(orientation="h", y=-0.12))
        fig.show()

    # Multi-date overlay: today, 3m, 6m, 1yr ago
    last_dt = ttf_df.dropna(how="all").index[-1]
    ref_dates, ref_colors = {}, {"Today": "#6366f1", "3m ago": "#3b82f6",
                                  "6m ago": "#f97316", "1yr ago": "#94a3b8"}
    ref_dates["Today"] = last_dt
    for label, days in [("3m ago", 63), ("6m ago", 126), ("1yr ago", 252)]:
        try:
            ref_dates[label] = ttf_df.index[ttf_df.index <= last_dt - pd.Timedelta(days=days)][-1]
        except IndexError:
            pass

    fig2 = go.Figure()
    for label, dt in ref_dates.items():
        row = ttf_df.loc[dt]
        ys  = [float(row.get(c, np.nan)) for c in m12]
        fig2.add_trace(go.Scatter(x=list(range(1, len(m12) + 1)), y=ys,
                                   mode="lines+markers",
                                   name=f"{label} ({pd.Timestamp(dt).strftime('%d %b %Y')})",
                                   line=dict(color=ref_colors.get(label, "#999"), width=2),
                                   marker=dict(size=5)))
    fig2.update_layout(template="plotly_white", height=420,
                       title="TTF Forward Curve \u2014 Multi-Date Overlay",
                       xaxis=dict(title="Tenor", tickvals=list(range(1, 13)),
                                  ticktext=[f"M{i}" for i in range(1, 13)]),
                       yaxis_title="\u20ac/MWh",
                       legend=dict(orientation="h", y=-0.15))
    fig2.show()

    if has_shape:
        print(f"Latest slope    (M1\u2013M12) : \u20ac{curve_df['slope'].iloc[-1]:.2f}/MWh")
        print(f"Latest curvature (M6 belly): \u20ac{curve_df['curvature'].iloc[-1]:.2f}/MWh")


## 4 · Seasonal Analysis

Box-plot distribution and average M1 price by calendar month. Average 21-day price change by month reveals seasonal buying/selling patterns.

In [ ]:
if ttf_df.empty:
    print("\u26a0\ufe0f  No TTF data.")
else:
    s = ttf_df["M1"].dropna()
    MN = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
    monthly = pd.DataFrame({"price": s, "month": s.index.month})

    # Box plot by month
    fig = go.Figure()
    for m in range(1, 13):
        grp = monthly[monthly["month"] == m]["price"]
        fig.add_trace(go.Box(y=grp, name=MN[m - 1], boxmean=True,
                             marker_color=px.colors.qualitative.Pastel[m - 1]))
    fig.update_layout(template="plotly_white", height=460,
                      title="TTF M1 Price Distribution by Calendar Month",
                      yaxis_title="\u20ac/MWh", showlegend=False)
    fig.show()

    # Average price bar
    avg_by_month = monthly.groupby("month")["price"].mean()
    fig2 = go.Figure(go.Bar(
        x=[MN[i - 1] for i in avg_by_month.index],
        y=avg_by_month.values,
        marker_color=px.colors.qualitative.Pastel[:12],
    ))
    fig2.update_layout(template="plotly_white", height=360,
                       title="Average TTF M1 Price by Calendar Month",
                       yaxis_title="\u20ac/MWh", xaxis_title="Month", showlegend=False)
    fig2.show()

    # MoM seasonality: 21d % change
    mom = s.pct_change(21) * 100
    monthly_ret = pd.DataFrame({"ret": mom, "month": mom.index.month})
    avg_ret = monthly_ret.groupby("month")["ret"].mean()
    fig3 = go.Figure(go.Bar(
        x=[MN[i - 1] for i in avg_ret.index],
        y=avg_ret.values,
        marker_color=["#22c55e" if v >= 0 else "#ef4444" for v in avg_ret],
    ))
    fig3.add_hline(y=0, line_color="black", line_width=0.8)
    fig3.update_layout(template="plotly_white", height=360,
                       title="Average 21-Day Price Change by Calendar Month",
                       yaxis_title="Avg % change", xaxis_title="Month", showlegend=False)
    fig3.show()

    best_m  = avg_by_month.idxmax()
    worst_m = avg_by_month.idxmin()
    print(f"Highest avg price: {MN[best_m - 1]}  (\u20ac{avg_by_month[best_m]:.1f}/MWh)")
    print(f"Lowest  avg price: {MN[worst_m - 1]}  (\u20ac{avg_by_month[worst_m]:.1f}/MWh)")


## 5 · Term Structure Carry & Roll Yield

Roll yield is the annualised return earned by holding M1 and rolling into M2 each month. Positive = backwardation (storage draws → earn carry). Negative = contango (injection season → pay to hold).

In [ ]:
if ttf_df.empty or "M1" not in ttf_df.columns or "M2" not in ttf_df.columns:
    print("\u26a0\ufe0f  Need at least M1 and M2 columns.")
else:
    MN = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]

    # M1-M2 roll yield: annualised return from the spread relative to M1
    m1m2 = (ttf_df["M1"] - ttf_df["M2"]).dropna()
    roll_yield = (m1m2 / ttf_df["M1"].reindex(m1m2.index)) * (252 / 21) * 100

    # Cumulative roll yield (daily share)
    cum_roll = (roll_yield / 252).cumsum()

    fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                        subplot_titles=["M1\u2013M2 Roll Yield (% annualised)",
                                        "Cumulative Roll Yield (%)"],
                        vertical_spacing=0.08, row_heights=[0.45, 0.55])

    fig.add_trace(go.Scatter(x=roll_yield.index, y=roll_yield.clip(lower=0),
                             fill="tozeroy", name="Backwardation (earn)",
                             line=dict(color="#22c55e", width=0.5),
                             fillcolor="rgba(34,197,94,0.28)"), row=1, col=1)
    fig.add_trace(go.Scatter(x=roll_yield.index, y=roll_yield.clip(upper=0),
                             fill="tozeroy", name="Contango (cost)",
                             line=dict(color="#ef4444", width=0.5),
                             fillcolor="rgba(239,68,68,0.28)"), row=1, col=1)
    fig.add_hline(y=0, line_color="black", line_width=0.8, row=1, col=1)

    fig.add_trace(go.Scatter(x=cum_roll.index, y=cum_roll,
                             name="Cumulative", line=dict(color="#6366f1", width=1.5)), row=2, col=1)
    fig.add_hline(y=0, line_color="black", line_width=0.8, row=2, col=1)

    fig.update_yaxes(title_text="% ann.", row=1, col=1)
    fig.update_yaxes(title_text="Cum. %", row=2, col=1)
    fig.update_layout(height=520, template="plotly_white",
                      title="TTF M1 Roll Yield \u2014 Carry from Rolling the Front Contract",
                      legend=dict(orientation="h", y=-0.12))
    fig.show()

    # Seasonal carry
    roll_m = pd.DataFrame({"roll": roll_yield, "month": roll_yield.index.month})
    avg_roll_m = roll_m.groupby("month")["roll"].mean()
    fig2 = go.Figure(go.Bar(
        x=[MN[i - 1] for i in avg_roll_m.index],
        y=avg_roll_m.values,
        marker_color=["#22c55e" if v >= 0 else "#ef4444" for v in avg_roll_m],
    ))
    fig2.add_hline(y=0, line_color="black", line_width=0.8)
    fig2.update_layout(template="plotly_white", height=360,
                       title="Average M1 Roll Yield by Calendar Month (Seasonal Carry)",
                       yaxis_title="% ann.", xaxis_title="Month", showlegend=False)
    fig2.show()

    pct_back = float((roll_yield > 0).mean() * 100)
    print(f"Average M1 roll yield   : {float(roll_yield.mean()):+.1f}% ann.")
    print(f"% days in backwardation : {pct_back:.1f}%")
    print(f"Latest roll yield       : {float(roll_yield.iloc[-1]):+.1f}% ann.")


## 6 · Spread Analysis vs Fundamentals

Deviation of M1 from its day-of-year seasonal average. Price by storage fill-rate quartile. Optional LNG correlation if `eu_lng_full.parquet` is available. Price elasticity to fill rate by season.

In [ ]:
if ttf_df.empty:
    print("\u26a0\ufe0f  No TTF data.")
else:
    s = ttf_df["M1"].dropna()

    # Seasonal norm: day-of-year average across all years
    s_df = s.to_frame("price")
    s_df["doy"] = s_df.index.dayofyear
    seas_norm = s_df.groupby("doy")["price"].mean()
    deviation = s - s_df["doy"].map(seas_norm).values

    # Storage quartile analysis
    if not storage_df.empty and "full" in storage_df.columns:
        merged = s.to_frame("M1").join(storage_df[["full"]], how="inner").dropna()
        merged["fill_q"] = pd.qcut(merged["full"], 4,
                                   labels=["Q1 low fill", "Q2", "Q3", "Q4 high fill"])
        q_avg = merged.groupby("fill_q", observed=True)["M1"].agg(["mean","median","count"])

        fig = make_subplots(rows=1, cols=2,
                            subplot_titles=["M1 Deviation from Seasonal Avg (\u20ac/MWh)",
                                            "Avg M1 Price by Storage Fill Quartile"])
        fig.add_trace(go.Scatter(x=deviation.index, y=deviation.clip(lower=0),
                                 fill="tozeroy", name="Above norm",
                                 line=dict(color="#ef4444", width=0.5),
                                 fillcolor="rgba(239,68,68,0.22)"), row=1, col=1)
        fig.add_trace(go.Scatter(x=deviation.index, y=deviation.clip(upper=0),
                                 fill="tozeroy", name="Below norm",
                                 line=dict(color="#22c55e", width=0.5),
                                 fillcolor="rgba(34,197,94,0.22)"), row=1, col=1)
        fig.add_hline(y=0, line_color="black", line_width=0.8, row=1, col=1)

        fig.add_trace(go.Bar(x=q_avg.index.astype(str), y=q_avg["mean"],
                             marker_color=["#ef4444","#f97316","#3b82f6","#22c55e"],
                             name="Avg M1 (\u20ac/MWh)"), row=1, col=2)
        fig.update_xaxes(tickangle=15, row=1, col=2)
        fig.update_layout(height=430, template="plotly_white",
                          title="TTF vs Seasonal Norm & Storage Quartiles",
                          legend=dict(orientation="h", y=-0.15))
        fig.show()

        print("Average M1 price by storage fill quartile:")
        for idx_q, row_q in q_avg.iterrows():
            print(f"  {str(idx_q):<18}: mean=\u20ac{row_q['mean']:.1f}  "
                  f"median=\u20ac{row_q['median']:.1f}  n={int(row_q['count'])}")

    # LNG vs price (optional)
    if not lng_df.empty and "full" in lng_df.columns:
        merged_lng = s.to_frame("M1").join(
            lng_df[["full"]].rename(columns={"full": "lng_full"}), how="inner").dropna()
        corr_lng = merged_lng["M1"].corr(merged_lng["lng_full"])
        fig3 = go.Figure(go.Scatter(x=merged_lng["lng_full"], y=merged_lng["M1"],
                                    mode="markers",
                                    marker=dict(size=3, opacity=0.4, color="#0ea5e9")))
        fig3.update_layout(template="plotly_white", height=380,
                           title=f"EU LNG Fill Rate vs TTF M1  (Pearson r = {corr_lng:.2f})",
                           xaxis_title="LNG Fill Rate (%)", yaxis_title="TTF M1 (\u20ac/MWh)")
        fig3.show()
    else:
        print("\u2139\ufe0f  eu_lng_full.parquet not found \u2014 LNG section skipped.")

    # Price elasticity by season
    if not storage_df.empty and "full" in storage_df.columns:
        m2 = s.to_frame("M1").join(storage_df[["full"]], how="inner").dropna()
        m2["fill_chg"]  = m2["full"].diff()
        m2["price_chg"] = m2["M1"].pct_change() * 100
        m2["season"]    = m2.index.month.map(lambda m: "Summer" if 4 <= m <= 9 else "Winter")
        m2c = m2.dropna()
        print("\nPrice elasticity (% price change per 1pp fill change) by season:")
        for season in ["Summer", "Winter"]:
            grp = m2c[m2c["season"] == season]
            if len(grp) > 10:
                sl, _, r, p, _ = stats.linregress(grp["fill_chg"], grp["price_chg"])
                print(f"  {season}: {sl:.3f}% per pp fill  (R\u00b2={r**2:.3f}  p={p:.4f}  n={len(grp)})")


## 7 · Risk Metrics

Historical P&L distribution with VaR 95/99. Maximum drawdown from peak. Annual Sharpe-like ratio and count of tail-move days (\|daily return\| > 5% or 10%).

In [ ]:
if ttf_df.empty:
    print("\u26a0\ufe0f  No TTF data.")
else:
    s = ttf_df["M1"].dropna()
    daily_ret = s.pct_change().dropna() * 100

    var95 = float(np.percentile(daily_ret, 5))
    var99 = float(np.percentile(daily_ret, 1))
    max_dd = float(((s - s.cummax()) / s.cummax() * 100).min())
    drawdown = (s - s.cummax()) / s.cummax() * 100

    fig = make_subplots(rows=1, cols=2,
                        subplot_titles=["Daily Return Distribution (%)",
                                        "Maximum Drawdown from Peak (%)"])
    fig.add_trace(go.Histogram(x=daily_ret, nbinsx=80, name="Daily return",
                               marker_color="#6366f1", opacity=0.75), row=1, col=1)
    fig.add_vline(x=var95, line_color="#f97316", line_dash="dash",
                  annotation_text=f"VaR95: {var95:.1f}%",
                  annotation_font_size=9, row=1, col=1)
    fig.add_vline(x=var99, line_color="#dc2626", line_dash="dash",
                  annotation_text=f"VaR99: {var99:.1f}%",
                  annotation_font_size=9, row=1, col=1)
    fig.add_trace(go.Scatter(x=drawdown.index, y=drawdown,
                             fill="tozeroy", name="Drawdown",
                             line=dict(color="#dc2626", width=1),
                             fillcolor="rgba(220,38,38,0.18)"), row=1, col=2)
    fig.update_yaxes(title_text="# days", row=1, col=1)
    fig.update_yaxes(title_text="Drawdown %", row=1, col=2)
    fig.update_xaxes(title_text="Return (%)", row=1, col=1)
    fig.update_layout(height=430, template="plotly_white",
                      title="TTF M1 \u2014 Return Distribution & Drawdown",
                      showlegend=False)
    fig.show()

    # Sharpe-like ratio and tail risk by year
    yr_df = pd.DataFrame({"ret": daily_ret, "year": daily_ret.index.year})
    years, sharpes, t5, t10 = [], [], [], []
    for yr, grp in yr_df.groupby("year"):
        r = grp["ret"]
        years.append(yr)
        sharpes.append(float(r.mean() / r.std() * np.sqrt(252)) if r.std() > 0 else 0)
        t5.append(int((r.abs() > 5).sum()))
        t10.append(int((r.abs() > 10).sum()))

    fig2 = make_subplots(rows=1, cols=2,
                         subplot_titles=["Sharpe-like Ratio by Year",
                                         "Tail Risk Days by Year"])
    fig2.add_trace(go.Bar(x=years, y=sharpes,
                          marker_color=["#22c55e" if v >= 0 else "#ef4444" for v in sharpes],
                          name="Sharpe"), row=1, col=1)
    fig2.add_hline(y=0, line_color="black", line_width=0.8, row=1, col=1)
    fig2.add_trace(go.Bar(x=years, y=t5, name=">5% move",
                          marker_color="#f97316", opacity=0.85), row=1, col=2)
    fig2.add_trace(go.Bar(x=years, y=t10, name=">10% move",
                          marker_color="#dc2626", opacity=0.85), row=1, col=2)
    fig2.update_layout(height=400, template="plotly_white",
                       title="TTF M1 Risk \u2014 Sharpe & Tail Events by Year",
                       barmode="overlay", legend=dict(orientation="h", y=-0.15))
    fig2.show()

    print(f"VaR 95%         : {var95:.2f}% daily")
    print(f"VaR 99%         : {var99:.2f}% daily")
    print(f"Max drawdown    : {max_dd:.1f}%")
    print(f"Days |ret|>5%   : {int((daily_ret.abs() > 5).sum())}")
    print(f"Days |ret|>10%  : {int((daily_ret.abs() > 10).sum())}")
    print("\nSharpe-like ratio by year:")
    for yr, sh in zip(years, sharpes):
        bar = "\u2588" * min(int(abs(sh * 5)), 20)
        print(f"  {yr}: {sh:+.2f}  {bar}")
